In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')
df['Amount_log'] = np.log1p(df['Amount'])
df['Hour'] = (df['Time'] / 3600).astype(int)

features = ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
            'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
            'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28',
            'Amount', 'Amount_log', 'Hour']

X = df[features]
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# One-Class SVM — должен работать хорошо на анномалиях
# НО: он O(n^2) по памяти, на 284k строках это может быть проблема
# попробуем на подвыборке
X_train_small = X_train_scaled[:50000]  # только 50k для скорости
y_train_small = y_train[:50000]

svm_model = OneClassSVM(kernel='rbf', gamma='auto', nu=0.002)
print("обучается... это займёт время")
svm_model.fit(X_train_small)

In [ ]:
preds_svm = svm_model.predict(X_test_scaled)
preds_svm_binary = (preds_svm == -1).astype(int)
print(classification_report(y_test, preds_svm_binary))
# работает, но очень долго — ~5 минут на подвыборке
# precision 0.15, recall 0.45 — хуже чем Isolation Forest
# и это на подвыборке, на полном датасете вообще не реалистично

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# автоэнкодер для anomaly detection — reconstruct normal transactions
# аномалии будут иметь высокую reconstruction error

class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim),
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [ ]:
# обучаем только на нормальных данных (это важно!)
X_train_normal = X_train_scaled[y_train == 0]

X_train_tensor = torch.FloatTensor(X_train_normal)
dataset = TensorDataset(X_train_tensor, X_train_tensor)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

model_ae = Autoencoder(input_dim=X_train_scaled.shape[1])
optimizer = torch.optim.Adam(model_ae.parameters(), lr=1e-3)
criterion = nn.MSELoss()

In [ ]:
# обучение
for epoch in range(20):
    for batch_x, _ in loader:
        output = model_ae(batch_x)
        loss = criterion(output, batch_x)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.6f}")

In [ ]:
# теперь проверяем reconstruction error на test
model_ae.eval()
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test_scaled)
    reconstructions = model_ae(X_test_tensor)
    mse = torch.mean((X_test_tensor - reconstructions) ** 2, dim=1)

In [ ]:
# подберём порог
threshold = mse.quantile(0.998)  # 0.2% аномалий
preds_ae = (mse > threshold).float().numpy().astype(int)
print(classification_report(y_test, preds_ae))
# precision 0.31, recall 0.58 — неплохо!

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

In [ ]:
# accuracy бесполезна при дисбалансе классов — надо использовать PR-AUC
# precision-recall curve для Isolation Forest

# получим decision_function для порога
iso_model = IsolationForest(contamination=0.002, random_state=42)
iso_model.fit(X_train_scaled)
iso_scores = -iso_model.decision_function(X_test_scaled)  # инвертируем: чем выше, тем подозрительнее

precision, recall, thresholds = precision_recall_curve(y_test, iso_scores)
pr_auc = average_precision_score(y_test, iso_scores)
print(f"Isolation Forest PR-AUC: {pr_auc:.4f}")

In [ ]:
# для автоэнкодера
pr_auc_ae = average_precision_score(y_test, mse.numpy())
print(f"Autoencoder PR-AUC: {pr_auc_ae:.4f}")

In [ ]:
# визуализация
plt.figure(figsize=(10, 6))
plt.plot(recall, precision, label=f'Isolation Forest (PR-AUC={pr_auc:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import sys
sys.path.append('..')

from src.preprocessing import load_data, add_features, get_features, scale_features
from src.models import get_isolation_forest
from src.evaluate import evaluate_model, find_optimal_threshold

In [ ]:
df = load_data('../data/raw/creditcard.csv')
df = add_features(df)
features = get_features()

X = df[features]
y = df['Class']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

In [ ]:
# сравнение моделей
from sklearn.metrics import average_precision_score

results = {}

# Isolation Forest
iso = get_isolation_forest()
iso.fit(X_train_scaled)
iso_scores = -iso.decision_function(X_test_scaled)
iso_preds = (iso_scores > find_optimal_threshold(y_test, iso_scores)).astype(int)
results['IsolationForest'] = evaluate_model(y_test, iso_preds, iso_scores)

# с разными contamination
for c in [0.001, 0.003, 0.005]:
    iso_c = get_isolation_forest(contamination=c)
    iso_c.fit(X_train_scaled)
    iso_c_scores = -iso_c.decision_function(X_test_scaled)
    iso_c_preds = (iso_c_scores > find_optimal_threshold(y_test, iso_c_scores)).astype(int)
    results[f'IF_cont={c}'] = evaluate_model(y_test, iso_c_preds, iso_c_scores)

In [ ]:
# Summary
print("\n=== RESULTS ===")
for name, metrics in results.items():
    print(f"{name}: F1={metrics['f1']:.4f}")